<a href="https://colab.research.google.com/github/nanubalasudheer-wq/MLA0204/blob/main/program3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import math
from collections import Counter
import os

# Calculate Entropy
def entropy(data):
    counts = Counter(data['target'])
    total = len(data)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())

# Calculate Information Gain
def information_gain(data, attribute):
    total_entropy = entropy(data)
    weighted_entropy = 0

    for value in data[attribute].unique():
        subset = data[data[attribute] == value]
        weighted_entropy += len(subset) / len(data) * entropy(subset)

    return total_entropy - weighted_entropy

# ID3 Algorithm
def id3(data, attributes):
    if data['target'].nunique() == 1:
        return data['target'].iloc[0]

    if not attributes:
        return data['target'].mode()[0]

    best = max(attributes, key=lambda a: information_gain(data, a))
    tree = {best: {}}

    for value in data[best].unique():
        subset = data[data[best] == value]
        remaining = [a for a in attributes if a != best]
        tree[best][value] = id3(subset, remaining)

    return tree

# Load Dataset (with fallback to default Play Tennis data if file is missing)
if os.path.exists('play_tennis.csv'):
    data = pd.read_csv('play_tennis.csv')
else:
    print("'play_tennis.csv' not found. Using the standard Play Tennis dataset...")
    raw_data = {
        'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain', 'Rain', 'Rain', 'Overcast', 'Sunny', 'Sunny', 'Rain', 'Sunny', 'Overcast', 'Overcast', 'Rain'],
        'Temperature': ['Hot', 'Hot', 'Hot', 'Mild', 'Cool', 'Cool', 'Cool', 'Mild', 'Cool', 'Mild', 'Mild', 'Mild', 'Hot', 'Mild'],
        'Humidity': ['High', 'High', 'High', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'Normal', 'Normal', 'High', 'Normal', 'High'],
        'Wind': ['Weak', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Weak', 'Weak', 'Strong', 'Strong', 'Weak', 'Strong'],
        'PlayTennis': ['No', 'No', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No']
    }
    data = pd.DataFrame(raw_data)

data.columns = data.columns.str.strip()
data.rename(columns={data.columns[-1]: 'target'}, inplace=True)

# Build Decision Tree
attributes = list(data.columns[:-1])
tree = id3(data, attributes)

print("\nDecision Tree:")
print(tree)

'play_tennis.csv' not found. Using the standard Play Tennis dataset...

Decision Tree:
{'Outlook': {'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}, 'Overcast': 'Yes', 'Rain': {'Wind': {'Weak': 'Yes', 'Strong': 'No'}}}}
